In [3]:
import numpy as np
import pandas as pd
import copy
cp = lambda x: copy.deepcopy(x)

def make_ordinal(n):
    # Check if the number ends in 11, 12, or 13
    if 11 <= (n % 100) <= 13:
        suffix = "th"
    else:
        # Match the last digit to the correct suffix
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"

In [4]:
# country-capital data
df_name = "country"
df = pd.read_csv("data/countries.csv")[["name", "capital"]].dropna()
print('read:', len(df))
df.head()

read: 245


,name,capital
0,Afghanistan,Kabul
1,Aland Islands,Mariehamn
2,Albania,Tirana
3,Algeria,Algiers
4,American Samoa,Pago Pago


In [9]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
torch.set_grad_enabled(False)

from matplotlib import pyplot as plt
import seaborn as sns

from general_utils import (
  ModelAndTokenizer,
  make_inputs,
  decode_tokens,
  find_token_range,
  predict_from_input,
)

from patchscopes_utils import *
from tqdm import tqdm
tqdm.pandas()

# for path:
from accelerate import Accelerator
from accelerate.utils import set_seed
import argparse
import pickle 
from transformers import AutoModelForCausalLM, AutoTokenizer

In [10]:
parser = argparse.ArgumentParser()
parser.add_argument("--exp_name", type=str)
parser.add_argument("--output_dir", type=str, default="./experiments/")
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--model_name", type=str, default="meta-llama/Llama-3.1-8B")

# args = parser.parse_args(args=['--exp_name', 'llama3.1', '--model_name', 'meta-llama/Llama-3.1-8B'])
# args = parser.parse_args(args=['--exp_name', 'pythia70m', '--model_name', 'EleutherAI/pythia-70m'])
# args = parser.parse_args(args=['--exp_name', 'pythia410m', '--model_name', 'EleutherAI/pythia-410m'])
# args = parser.parse_args(args=['--exp_name', 'gemma3.270m', '--model_name', 'google/gemma-3-270m'])
# args = parser.parse_args(args=['--exp_name', 'gemma2.2b', '--model_name', 'google/gemma-2-2b'])
# args = parser.parse_args(args=['--exp_name', 'gemma.2b', '--model_name', 'google/gemma-2b'])
# args = parser.parse_args(args=['--exp_name', 'gemma2.9b', '--model_name', 'google/gemma-2-9b'])
args = parser.parse_args(args=['--exp_name', 'phi2', '--model_name', 'microsoft/phi-2'])
# args = parser.parse_args(args=['--exp_name', 'phi1', '--model_name', 'microsoft/phi-1'])
# args = parser.parse_args(args=['--exp_name', 'qwen2.5.1.5b', '--model_name', 'Qwen/Qwen2.5-1.5B'])

print(args)

set_seed(args.seed)
output_dir = os.path.join(args.output_dir, args.exp_name)
os.makedirs(output_dir, exist_ok=True)

Namespace(exp_name='phi2', output_dir='./experiments/', seed=42, model_name='microsoft/phi-2')


In [11]:
model_to_hook = {
    "EleutherAI/pythia-6.9b": set_hs_patch_hooks_neox,
    "EleutherAI/pythia-12b": set_hs_patch_hooks_neox,
    "meta-llama/Llama-2-13b-hf": set_hs_patch_hooks_llama,
    "lmsys/vicuna-7b-v1.5": set_hs_patch_hooks_llama,
    "./stable-vicuna-13b": set_hs_patch_hooks_llama,
    "CarperAI/stable-vicuna-13b-delta": set_hs_patch_hooks_llama,
    "EleutherAI/gpt-j-6b": set_hs_patch_hooks_gptj,
    
    "EleutherAI/pythia-70m": set_hs_patch_hooks_neox,
    "EleutherAI/pythia-410m": set_hs_patch_hooks_neox,    
    "google/gemma-2-2b": set_hs_patch_hooks_llama,
    "google/gemma-2b": set_hs_patch_hooks_llama,
    "google/gemma-3-270m": set_hs_patch_hooks_llama,
    "google/gemma-2-9b": set_hs_patch_hooks_llama,
    "meta-llama/Llama-3.1-8B": set_hs_patch_hooks_llama,
    "Qwen/Qwen2.5-1.5B": set_hs_patch_hooks_llama,    
    "microsoft/phi-2": set_hs_patch_hooks_phi,
    "microsoft/phi-1": set_hs_patch_hooks_phi
}

model_to_head = {
    "Eleu": "embed_out",
    "goog": "lm_head",
    "meta": "lm_head",
    "micr": "lm_head",
    "Qwen": "lm_head"
}

In [12]:
# Load model

model_name = args.model_name # "microsoft/phi-2" # "EleutherAI/pythia-70m" # "microsoft/phi-2"
sos_tok = False

if "9b" in model_name or "8b" in model_name:
    torch_dtype = torch.bfloat16 # torch.float16
else:
    torch_dtype = None

my_device = torch.device("cuda:0") # mps cuda:1

mt = ModelAndTokenizer(
    model_name,
    low_cpu_mem_usage=False,
    torch_dtype=torch_dtype,
    device=my_device,
)
mt.set_hs_patch_hooks = model_to_hook[model_name]
mt.model.eval()

# NOTE: should check the name of the unembedding layer:

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 

In [ ]:
# OPTIONAL! sanity check.
prompt = "The capital of France is the city of"
inputs = mt.tokenizer(prompt,return_tensors="pt",).to(mt.model.device)
with torch.no_grad():
    outputs = mt.model(**inputs)

logits = outputs.logits[0, -1]

print("dtype:", logits.dtype)
print("has_nan:", torch.isnan(logits).any().item())
print("has_inf:", torch.isinf(logits).any().item())
print("max:", logits.float().max().item())
print("min:", logits.float().min().item())

probs = torch.softmax(logits.float(), dim=-1)

topk_probs, topk_ids = torch.topk(probs, 10)

for p, idx in zip(topk_probs, topk_ids):
    token = mt.tokenizer.convert_ids_to_tokens(int(idx))
    print(f"{token:20s} {p.item():.6f}")